# Phase 2.5 — CIC-IDS2018 Network Feature Integration
**Input:** CIC-IDS2018 CSV files + `df_identity_clean.csv`  
**Output:** `Dataset/Processed/features_enriched.csv` (replaces features.csv for Phase 3 onward)  

**What this notebook does:**
1. Load and sample the two CIC-IDS2018 files
2. Extract per-IP network behavioral profiles from traffic data
3. Map network risk scores onto the CloudTrail IP addresses
4. Merge enriched IP features into the existing feature matrix
5. Save enriched features for Phase 3 onward

**Why this matters for the thesis:**  
The original feature matrix used only identity-layer signals (IAM events, error rates).  
Adding network-layer signals (packet size, flow duration, attack label from CIC-IDS2018)  
makes the IP nodes in the HGNN graph carry richer features, improving anomaly detection.

## Cell 1 — Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

# update these paths to match your local setup
CIC_FRIDAY    = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Raw/CIC-IDS2018/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv")
CIC_WEDNESDAY = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Raw/CIC-IDS2018/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv")
PHASE1_PATH   = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv")
FEATURES_PATH = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features.csv")
OUTPUT_PATH   = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features_enriched.csv")

print("Paths set. Loading data...")

Paths set. Loading data...


## Cell 2 — Load & Sample CIC-IDS2018
These files are large (1-2 GB each). We sample 10,000 rows per file —  
stratified by Label so we keep a proportional mix of attack and benign traffic.

In [12]:
SAMPLE_PER_FILE = 10000

def load_cic_sample(filepath, n=SAMPLE_PER_FILE):
    print(f"Loading: {os.path.basename(filepath)}")
    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=50000, low_memory=False):
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True)
    print(f"  Full size: {len(df):,} rows")

    # Strip whitespace from Label values
    df['Label'] = df['Label'].astype(str).str.strip()

    # Sample per label class manually to avoid groupby dropping Label
    unique_labels = df['Label'].unique()
    n_per_class   = n // len(unique_labels)
    sampled_parts = []
    for label in unique_labels:
        subset = df[df['Label'] == label]
        sampled_parts.append(
            subset.sample(min(len(subset), n_per_class), random_state=42)
        )
    sampled = pd.concat(sampled_parts, ignore_index=True)

    print(f"  Sampled: {len(sampled):,} rows")
    print(f"  Label distribution:")
    print(sampled['Label'].value_counts().to_string())
    print(f"  Label column present: {'Label' in sampled.columns}")
    return sampled


df_friday    = load_cic_sample(CIC_FRIDAY)
df_wednesday = load_cic_sample(CIC_WEDNESDAY)

df_cic = pd.concat([df_friday, df_wednesday], ignore_index=True)
print(f"\nCombined CIC dataset: {len(df_cic):,} rows")

Loading: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Full size: 1,048,575 rows
  Sampled: 10,000 rows
  Label distribution:
Label
Benign    5000
Bot       5000
  Label column present: True
Loading: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  Full size: 1,048,575 rows
  Sampled: 9,999 rows
  Label distribution:
Label
Benign            3333
FTP-BruteForce    3333
SSH-Bruteforce    3333
  Label column present: True

Combined CIC dataset: 19,999 rows


In [28]:
df_cic.info()

<class 'pandas.DataFrame'>
Index: 19944 entries, 0 to 19998
Data columns (total 81 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Dst Port           19944 non-null  int64  
 1   Protocol           19944 non-null  int64  
 2   Timestamp          19944 non-null  str    
 3   Flow Duration      19944 non-null  int64  
 4   Tot Fwd Pkts       19944 non-null  int64  
 5   Tot Bwd Pkts       19944 non-null  int64  
 6   TotLen Fwd Pkts    19944 non-null  int64  
 7   TotLen Bwd Pkts    19944 non-null  float64
 8   Fwd Pkt Len Max    19944 non-null  int64  
 9   Fwd Pkt Len Min    19944 non-null  int64  
 10  Fwd Pkt Len Mean   19944 non-null  float64
 11  Fwd Pkt Len Std    19944 non-null  float64
 12  Bwd Pkt Len Max    19944 non-null  int64  
 13  Bwd Pkt Len Min    19944 non-null  int64  
 14  Bwd Pkt Len Mean   19944 non-null  float64
 15  Bwd Pkt Len Std    19944 non-null  float64
 16  Flow Byts/s        19944 non-null  flo

## Cell 3 — Clean CIC-IDS2018 Data
Remove infinite values and NaNs that are common in CICFlowMeter output.

In [13]:
# Standardize label column name
if ' Label' in df_cic.columns:
    df_cic.rename(columns={' Label': 'Label'}, inplace=True)
df_cic['Label'] = df_cic['Label'].str.strip()

# Replace infinite values with NaN then drop
df_cic.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df_cic)
df_cic.dropna(inplace=True)
after = len(df_cic)
print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}")
print(f"Rows dropped:         {before - after:,}")

# Create binary attack label: 0 = Benign, 1 = Attack
df_cic['is_network_attack'] = (df_cic['Label'] != 'Benign').astype(int)

print(f"\nNetwork attack label distribution:")
print(df_cic['is_network_attack'].value_counts().to_string())
print(f"\nAttack types:")
print(df_cic['Label'].value_counts().to_string())

Rows before cleaning: 19,999
Rows after cleaning:  19,944
Rows dropped:         55

Network attack label distribution:
is_network_attack
1    11666
0     8278

Attack types:
Label
Benign            8278
Bot               5000
FTP-BruteForce    3333
SSH-Bruteforce    3333


In [26]:
df_cic.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,is_network_attack
0,443,6,02/03/2018 02:12:32,115754899,19,16,1658,1205.0,1048,0,...,43521.66667,57261.07733,225350.0,26964.0,9602718.583,1361490.661,10000000.0,5279624.0,Benign,0
1,445,6,02/03/2018 10:29:40,1105669,6,5,455,338.0,140,0,...,0.00000,0.00000,0.0,0.0,0.000,0.000,0.0,0.0,Benign,0
2,53,17,02/03/2018 01:37:53,253,1,1,41,57.0,41,41,...,0.00000,0.00000,0.0,0.0,0.000,0.000,0.0,0.0,Benign,0
3,3389,6,02/03/2018 04:49:05,106162,3,1,0,0.0,0,0,...,0.00000,0.00000,0.0,0.0,0.000,0.000,0.0,0.0,Benign,0
4,53,17,02/03/2018 09:20:23,609,1,1,45,73.0,45,45,...,0.00000,0.00000,0.0,0.0,0.000,0.000,0.0,0.0,Benign,0


In [27]:
df_cic.info()

<class 'pandas.DataFrame'>
Index: 19944 entries, 0 to 19998
Data columns (total 81 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Dst Port           19944 non-null  int64  
 1   Protocol           19944 non-null  int64  
 2   Timestamp          19944 non-null  str    
 3   Flow Duration      19944 non-null  int64  
 4   Tot Fwd Pkts       19944 non-null  int64  
 5   Tot Bwd Pkts       19944 non-null  int64  
 6   TotLen Fwd Pkts    19944 non-null  int64  
 7   TotLen Bwd Pkts    19944 non-null  float64
 8   Fwd Pkt Len Max    19944 non-null  int64  
 9   Fwd Pkt Len Min    19944 non-null  int64  
 10  Fwd Pkt Len Mean   19944 non-null  float64
 11  Fwd Pkt Len Std    19944 non-null  float64
 12  Bwd Pkt Len Max    19944 non-null  int64  
 13  Bwd Pkt Len Min    19944 non-null  int64  
 14  Bwd Pkt Len Mean   19944 non-null  float64
 15  Bwd Pkt Len Std    19944 non-null  float64
 16  Flow Byts/s        19944 non-null  flo

## Cell 4 — Build Per-Port Network Risk Profile
CIC-IDS2018 does not have source IPs but has destination ports.  
We build a risk profile per destination port — attack rate, avg packet size, avg flow duration.  
These port-level statistics will then be mapped onto CloudTrail IPs via their associated services.

In [ ]:
# Network features to aggregate per destination port
NETWORK_FEATURE_COLS = [
    'Flow Duration',
    'Tot Fwd Pkts',
    'Tot Bwd Pkts',
    'TotLen Fwd Pkts',
    'TotLen Bwd Pkts',
    'Pkt Len Mean',
    'Pkt Len Std',
    'Flow Byts/s',
    'Flow Pkts/s',
    'is_network_attack',
]

port_profile = df_cic.groupby('Dst Port').agg(
    flow_count = ('Flow Duration', 'count'),
    avg_flow_duration = ('Flow Duration', 'mean'),
    avg_fwd_pkts = ('Tot Fwd Pkts', 'mean'),
    avg_bwd_pkts = ('Tot Bwd Pkts', 'mean'),
    avg_pkt_len = ('Pkt Len Mean', 'mean'),
    avg_pkt_std = ('Pkt Len Std', 'mean'),
    avg_flow_bytes_s = ('Flow Byts/s', 'mean'),
    avg_flow_pkts_s = ('Flow Pkts/s', 'mean'),
    network_attack_rate = ('is_network_attack', 'mean'),
).reset_index()

print(f"Port profiles built: {len(port_profile)} unique destination ports")
print(f"\nTop 10 highest risk ports (by attack rate):")
print(port_profile.nlargest(10, 'network_attack_rate')[['Dst Port', 'flow_count', 'network_attack_rate']].to_string())

Port profiles built: 1199 unique destination ports

Top 10 highest risk ports (by attack rate):
     Dst Port  flow_count  network_attack_rate
1          21        3333                  1.0
60       8080        4931                  1.0
591     50736           1                  1.0
593     50739           1                  1.0
597     50772           1                  1.0
598     50773           1                  1.0
603     50795           1                  1.0
607     50811           1                  1.0
615     50849           1                  1.0
623     50879           1                  1.0


## Cell 5 — Build Overall Network Risk Score
Since CloudTrail IPs are internal AWS IPs (not directly in CIC traffic),  
we derive a global network risk context score from the CIC dataset.  
This becomes additional features appended to each event row.

In [ ]:
# Compute global network statistics from CIC dataset
# These represent the threat landscape context during the same time period
global_network_stats = {
    'global_attack_rate': df_cic['is_network_attack'].mean(),
    'global_avg_flow_duration': df_cic['Flow Duration'].mean(),
    'global_avg_pkt_len': df_cic['Pkt Len Mean'].mean(),
    'global_avg_flow_bytes_s': df_cic['Flow Byts/s'].mean(),
    'global_bot_rate': (df_cic['Label'] == 'Bot').mean(),
    'global_bruteforce_rate': (df_cic['Label'] == 'FTP-BruteForce').mean(),
}

print("Global network risk context from CIC-IDS2018:")
for k, v in global_network_stats.items():
    print(f"  {k:<30}: {v:.6f}")

Global network risk context from CIC-IDS2018:
  global_attack_rate            : 0.584938
  global_avg_flow_duration      : 6444780.273265
  global_avg_pkt_len            : 53.037901
  global_avg_flow_bytes_s       : 146423.743551
  global_bot_rate               : 0.250702
  global_bruteforce_rate        : 0.167118


## Cell 6 — Map Network Features onto CloudTrail IPs
For each unique IP in the CloudTrail dataset, we assign network risk features  
based on the known port associated with their AWS service calls.  
AWS service port mappings: S3=443, IAM=443, KMS=443, STS=443, SSM=443,  
SecretsManager=443, EC2=443, CloudTrail=443, Route53=53, RDS=3306

In [ ]:
df_phase1    = pd.read_csv(PHASE1_PATH)
df_features  = pd.read_csv(FEATURES_PATH)

# AWS service -> typical destination port mapping
SERVICE_PORT_MAP = {
    's3.amazonaws.com': 443,
    'iam.amazonaws.com': 443,
    'sts.amazonaws.com': 443,
    'kms.amazonaws.com': 443,
    'secretsmanager.amazonaws.com': 443,
    'ssm.amazonaws.com': 443,
    'ec2.amazonaws.com': 443,
    'cloudtrail.amazonaws.com': 443,
    'logs.amazonaws.com': 443,
    'account.amazonaws.com': 443,
    'health.amazonaws.com': 443,
    'notifications.amazonaws.com': 443,
    'organizations.amazonaws.com': 443,
    'route53.amazonaws.com': 53,
    'rds.amazonaws.com': 3306,
}

# Map each event's service to a port, then look up network stats for that port
df_phase1['mapped_port'] = df_phase1['eventSource'].map(SERVICE_PORT_MAP).fillna(443).astype(int)

# Merge port-level network profile onto each event
df_phase1 = df_phase1.merge(
    port_profile[['Dst Port', 'avg_flow_duration', 'avg_pkt_len',
                  'avg_flow_bytes_s', 'network_attack_rate']],
    left_on='mapped_port',
    right_on='Dst Port',
    how='left'
)

# For ports not found in CIC data, fill with global averages
df_phase1['avg_flow_duration'].fillna(global_network_stats['global_avg_flow_duration'], inplace=True)
df_phase1['avg_pkt_len'].fillna(global_network_stats['global_avg_pkt_len'], inplace=True)
df_phase1['avg_flow_bytes_s'].fillna(global_network_stats['global_avg_flow_bytes_s'], inplace=True)
df_phase1['network_attack_rate'].fillna(global_network_stats['global_attack_rate'], inplace=True)

# Add global context features to every event row
df_phase1['global_bot_rate']        = global_network_stats['global_bot_rate']
df_phase1['global_bruteforce_rate'] = global_network_stats['global_bruteforce_rate']

print("Network features mapped onto CloudTrail events.")
print(f"\nNew network columns added:")
new_cols = ['avg_flow_duration', 'avg_pkt_len', 'avg_flow_bytes_s',
            'network_attack_rate', 'global_bot_rate', 'global_bruteforce_rate']
print(df_phase1[new_cols].describe().round(4))

Network features mapped onto CloudTrail events.

New network columns added:
       avg_flow_duration  avg_pkt_len  avg_flow_bytes_s  network_attack_rate  \
count       9.930000e+02     993.0000          993.0000                993.0   
mean        4.720637e+07     214.4291       656338.4926                  0.0   
std         0.000000e+00       0.0000            0.0000                  0.0   
min         4.720637e+07     214.4291       656338.4926                  0.0   
25%         4.720637e+07     214.4291       656338.4926                  0.0   
50%         4.720637e+07     214.4291       656338.4926                  0.0   
75%         4.720637e+07     214.4291       656338.4926                  0.0   
max         4.720637e+07     214.4291       656338.4926                  0.0   

       global_bot_rate  global_bruteforce_rate  
count        1000.0000               1000.0000  
mean            0.2507                  0.1671  
std             0.0000                  0.0000  
min    

/var/folders/s5/ck6frb1j5sq98srb5ml9wg9m0000gn/T/ipykernel_10233/2155948899.py:36: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_phase1['avg_flow_duration'].fillna(global_network_stats['global_avg_flow_duration'], inplace=True)
/var/folders/s5/ck6frb1j5sq98srb5ml9wg9m0000gn/T/ipykernel_10233/2155948899.py:37: ChainedAssignmentErro

## Cell 7 — Normalize New Network Features
Apply MinMaxScaler to the new network features so they are on the same [0,1] scale  
as the existing behavioral profile features from Phase 2.

In [17]:
NETWORK_COLS_TO_NORMALIZE = [
    'avg_flow_duration',
    'avg_pkt_len',
    'avg_flow_bytes_s',
    'network_attack_rate',
    'global_bot_rate',
    'global_bruteforce_rate',
]

scaler = MinMaxScaler()
df_phase1[NETWORK_COLS_TO_NORMALIZE] = scaler.fit_transform(
    df_phase1[NETWORK_COLS_TO_NORMALIZE]
)

print("Normalized network features (all values should be between 0.0 and 1.0):")
print(df_phase1[NETWORK_COLS_TO_NORMALIZE].describe().round(4))

Normalized network features (all values should be between 0.0 and 1.0):
       avg_flow_duration  avg_pkt_len  avg_flow_bytes_s  network_attack_rate  \
count              993.0        993.0             993.0                993.0   
mean                 0.0          0.0               0.0                  0.0   
std                  0.0          0.0               0.0                  0.0   
min                  0.0          0.0               0.0                  0.0   
25%                  0.0          0.0               0.0                  0.0   
50%                  0.0          0.0               0.0                  0.0   
75%                  0.0          0.0               0.0                  0.0   
max                  0.0          0.0               0.0                  0.0   

       global_bot_rate  global_bruteforce_rate  
count           1000.0                  1000.0  
mean               0.0                     0.0  
std                0.0                     0.0  
min        

## Cell 8 — Merge into Final Enriched Feature Matrix
Combine the original Phase 2 features with the new network features.

In [18]:
# The network features are aligned by row index with df_features
# (both are derived from the same 1000-row df_identity_clean.csv)
network_feature_cols = [
    'avg_flow_duration',
    'avg_pkt_len',
    'avg_flow_bytes_s',
    'network_attack_rate',
    'global_bot_rate',
    'global_bruteforce_rate',
]

# Reset index to ensure clean concat
df_features = df_features.reset_index(drop=True)
df_network  = df_phase1[network_feature_cols].reset_index(drop=True)

df_enriched = pd.concat([df_features, df_network], axis=1)

print(f"Original feature matrix shape : {df_features.shape}")
print(f"Enriched feature matrix shape : {df_enriched.shape}")
print(f"New columns added             : {len(network_feature_cols)}")
print(f"\nFinal columns:")
for col in df_enriched.columns:
    print(f"  - {col}")

Original feature matrix shape : (1000, 23)
Enriched feature matrix shape : (1000, 29)
New columns added             : 6

Final columns:
  - userName_enc
  - sourceIPAddress_enc
  - eventSource_enc
  - eventName_enc
  - type_enc
  - sessionContext.attributes.mfaAuthenticated_enc
  - sessionContext.sessionIssuer.type_enc
  - errorCode_enc
  - eventType_enc
  - readOnly_enc
  - event_hour
  - event_dayofweek
  - is_weekend
  - is_off_hours
  - action_count
  - unique_ips
  - unique_services
  - unique_events
  - error_rate
  - off_hours_rate
  - weekend_rate
  - readonly_ratio
  - is_anomaly
  - avg_flow_duration
  - avg_pkt_len
  - avg_flow_bytes_s
  - network_attack_rate
  - global_bot_rate
  - global_bruteforce_rate


## Cell 9 — Sanity Check
Verify label distribution is preserved and no missing values were introduced.

In [19]:
print("Missing values in enriched matrix:")
missing = df_enriched.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "  None — all clean.")

print("\nLabel distribution (must match Phase 2):")
label_counts = df_enriched['is_anomaly'].value_counts()
total = len(df_enriched)
print(f"  Normal   (0): {label_counts.get(0, 0):>5} rows  ({label_counts.get(0,0)/total*100:.1f}%)")
print(f"  Anomaly  (1): {label_counts.get(1, 0):>5} rows  ({label_counts.get(1,0)/total*100:.1f}%)")

print("\nSample of enriched rows (network columns):")
df_enriched[['userName_enc', 'eventName_enc', 'is_anomaly'] + network_feature_cols].head(5)

Missing values in enriched matrix:
avg_flow_duration      7
avg_pkt_len            7
avg_flow_bytes_s       7
network_attack_rate    7
dtype: int64

Label distribution (must match Phase 2):
  Normal   (0):   936 rows  (93.6%)
  Anomaly  (1):    64 rows  (6.4%)

Sample of enriched rows (network columns):


,userName_enc,eventName_enc,is_anomaly,avg_flow_duration,avg_pkt_len,avg_flow_bytes_s,network_attack_rate,global_bot_rate,global_bruteforce_rate
0,2,18,0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,116,0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,135,0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,124,0,0.0,0.0,0.0,0.0,0.0,0.0
4,2,2,0,0.0,0.0,0.0,0.0,0.0,0.0


## Cell 10 — Save Enriched Feature Matrix

In [20]:
df_enriched.to_csv(OUTPUT_PATH, index=False)

print(f"Saved enriched features to: {OUTPUT_PATH}")
print(f"Shape: {df_enriched.shape}")
print()
print("NEXT STEPS:")
print("  Phase 3 onward must use features_enriched.csv instead of features.csv")
print("  Update FEATURES_PATH in Phase 3 Cell 2 and Phase 5 Cell 1 to point to:")
print(f"  {OUTPUT_PATH}")

Saved enriched features to: /Users/philberttan/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features_enriched.csv
Shape: (1000, 29)

NEXT STEPS:
  Phase 3 onward must use features_enriched.csv instead of features.csv
  Update FEATURES_PATH in Phase 3 Cell 2 and Phase 5 Cell 1 to point to:
  /Users/philberttan/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features_enriched.csv
